# FDA Approval Speed and ARCOS Feasibility — Exploratory Pipeline**Project:** ECON 580 Thesis (Regulatory speed and diversion risk)  **Date:** 2026-03-08  This notebook builds a reproducible exploratory pipeline to understand FDA approval data relevant to regulatory speed and to scope the feasibility of downstream ARCOS linkage. It is **not** a causal analysis.

## Objective- Assemble a clean FDA backbone dataset focused on original approvals.- Document what variables are available and what they do **not** support.- Produce exploratory summaries (counts, trends, and simple visuals).- Create ARCOS merge scaffolding and document feasibility limitations.

## Data Sources (Accessed 2026-03-08)- **FDA Drugs@FDA Data Files** (official data download): https://www.fda.gov/drugs/drug-approvals-and-databases/drugsfda-data-files- **FDA Drugs@FDA zip (direct download)**: https://www.fda.gov/media/89850/download?attachment- **NBER mirror** (fallback if FDA download fails): https://www.nber.org/research/data/fda-drugsfda-data-filesNotes:- FDA provides data files as tab-delimited tables inside a zip archive.- The data include *approval/tentative approval dates* but **do not include submission dates**, which limits direct review-duration measurement.

## Reproducibility Notes- Paths are handled with `pathlib` and relative to the project root.- Downloads are cached in `data/raw/` to avoid repeated calls.- If the FDA download fails, place the zip manually at `data/raw/drugsatfda.zip` and rerun.

In [ ]:
from pathlib import Pathimport sysimport pandas as pdimport numpy as npimport matplotlib.pyplot as plt# Local module pathROOT = Path("..").resolve()sys.path.append(str(ROOT / "src"))from fda_pipeline import (    ensure_dirs,    download_fda_zip,    read_fda_table,    build_backbone,    build_data_dictionary,    create_summary_tables,    save_summary_tables,    make_plots,    create_arcos_stub,    FDA_ZIP_URL,)pd.set_option("display.max_columns", 50)

In [ ]:
# Set up project directoriespaths = ensure_dirs(ROOT)paths

## Data IngestionWe download the FDA Drugs@FDA zip file (if not already cached), then load key tables.

In [ ]:
zip_path = paths.raw_dir / "drugsatfda.zip"zip_path = download_fda_zip(zip_path, log_path=paths.logs_dir / "download_failures.log")if zip_path is None or not zip_path.exists():    raise RuntimeError(        "FDA download failed. Place the zip at data/raw/drugsatfda.zip and rerun."    )zip_path

### Load FDA TablesKey tables used in this exploratory build:- `Applications.txt`: application-level information (application type, sponsor).- `Submissions.txt`: submission-level information (approval status/date, review priority).- `Products.txt`: product details (drug name, active ingredient, dosage form).- `SubmissionClass_Lookup.txt`: submission class definitions.- `SubmissionPropertyType.txt`: includes **orphan designation** flags (limited).

In [ ]:
applications = read_fda_table(zip_path, "Applications.txt")submissions = read_fda_table(zip_path, "Submissions.txt")products = read_fda_table(zip_path, "Products.txt")submission_class_lookup = read_fda_table(zip_path, "SubmissionClass_Lookup.txt")submission_properties = read_fda_table(zip_path, "SubmissionPropertyType.txt"){    "applications": applications.shape,    "submissions": submissions.shape,    "products": products.shape,    "submission_class_lookup": submission_class_lookup.shape,    "submission_properties": submission_properties.shape,}

## Build the FDA Backbone DatasetWe focus on **ORIG** submissions to approximate *original approvals*. We merge:- application info (sponsor, application type)- submission class lookup- product info (aggregated by application)- orphan designation flag (when present)**Important limitation:** The FDA data files **do not include submission dates**, so review-duration metrics cannot be computed directly here. We keep placeholder columns for future merges.

In [ ]:
backbone = build_backbone(    applications,    submissions,    products,    submission_class_lookup,    submission_properties,)backbone.head()

### Save Processed OutputsWe save:- `data/processed/fda_backbone.csv`- `data/processed/fda_backbone_data_dictionary.csv`

In [ ]:
backbone_path = paths.processed_dir / "fda_backbone.csv"backbone.to_csv(backbone_path, index=False)data_dictionary = build_data_dictionary()data_dictionary_path = paths.processed_dir / "fda_backbone_data_dictionary.csv"data_dictionary.to_csv(data_dictionary_path, index=False)(backbone_path, data_dictionary_path)

## Validation and Data Quality ChecksWe check:- Row counts before/after filtering to ORIG submissions- Duplicates in the backbone primary key- Missingness by key variables- Approval date ranges

In [ ]:
# Row countsorig_rows = submissions[submissions["SubmissionType"].astype(str).str.strip() == "ORIG"].shape[0]print("ORIG submissions in raw table:", orig_rows)print("Backbone rows:", backbone.shape[0])# Duplicate check on appl_no + submission_nodup_count = backbone.duplicated(subset=["appl_no", "submission_no"]).sum()print("Duplicate appl_no + submission_no rows:", dup_count)# Missingness (top 10)missingness = backbone.isna().mean().sort_values(ascending=False).head(10)missingness

In [ ]:
# Approval date rangeprint("Approval date range:")print(backbone["approval_date"].min(), "to", backbone["approval_date"].max())# Review duration availabilityprint("Review duration missingness (%):", backbone["review_duration_days"].isna().mean() * 100)

## Descriptive ExplorationWe compute summary tables and simple plots:- approvals per year- review priority counts by year- orphan designation counts by year- top active ingredients (by appearance)Plots are saved to `output/figures/` and tables to `output/tables/`.

In [ ]:
tables = create_summary_tables(backbone)save_summary_tables(tables, paths.tables_dir)make_plots(backbone, tables, paths.figures_dir)# Show a few tablesprint(tables["approvals_by_year"].head())print(tables["priority_by_year"].head())print(tables["top_active_ingredients"].head())

## ARCOS Feasibility / Pilot Merge ScaffoldingARCOS captures **controlled-substance distribution**, but the FDA data files do not include DEA schedules. We therefore:- Create a stub mapping table of **unique active ingredients** for manual classification.- Save a candidate file with approval-level details and normalized ingredient text for future joins.If ARCOS data are later added locally, those can be matched using the normalized ingredient names.

In [ ]:
# Search for ARCOS files locally (optional)arcos_files = [p for p in (ROOT / "data").rglob("*arcos*") if p.is_file()]print("ARCOS files found:", arcos_files[:5])create_arcos_stub(backbone, paths.intermediate_dir)stub_path = paths.intermediate_dir / "controlled_substance_mapping_stub.csv"candidates_path = paths.intermediate_dir / "arcos_merge_candidates.csv"(pd.read_csv(stub_path).head(), pd.read_csv(candidates_path).head())

## Limitations- **Submission dates are not in Drugs@FDA data files**, so review duration (submission-to-approval time) cannot be calculated here.- Approval date is an administrative action date, **not market launch**.- Expedited program flags beyond orphan designation (fast track, breakthrough therapy, accelerated approval) are **not present** in this FDA download.- ARCOS linkage requires external controlled-substance schedule data or ARCOS reference tables.

## Exploratory Findings and Recommended Next Steps**What was assembled:**- A clean FDA backbone dataset (one row per ORIG submission) with approval dates, review priority, sponsor, and product/ingredient information.- Summary tables and figures showing approval counts and review priority trends over time.- ARCOS merge scaffolding (ingredient mapping stub + candidate file).**Usable variables right now:**- Approval date (administrative action date).- Review priority (PRIORITY vs STANDARD vs other/unknown).- Orphan designation (limited flag).- Sponsor name, application type, drug name, active ingredient.**Regulatory speed feasibility:**- True review duration **cannot be computed** from this FDA file alone because submission dates are missing.- Priority vs standard review is currently the best observable proxy for “speed,” but it is not identical to review duration.**ARCOS linkage feasibility:**- Feasible only after obtaining controlled-substance schedule data or ARCOS reference tables.- Active ingredient names are present and normalized for future matching, but require manual verification.**Recommended next steps:**- Use priority vs standard review as the primary speed proxy **for now** and document its limitations.- Acquire official DEA schedule lists or ARCOS reference files to flag controlled substances.- Keep expedited program flags separate once additional FDA datasets are located.- Avoid causal claims until submission dates and downstream distribution measures are properly merged.